# Shared Libraries in Windows and Linux

Prise de notes sur les librairies partagées dans Windows et Linux a partir de la conférance de [Ofek Shilon](https://www.youtube.com/watch?v=6TrJc06IekE).

## Introduction

Au début de sa présentation, Ofek Shilon explique qu'il travaillé principalement sur Windows pendant sa carrière, mais il est passé à Linux il y a environ quatre ans. La transition a été difficile, en particulier en ce qui concerne l'éditeur de liens (linker). Dans cette présentation, il discutera des différences entre Windows et Linux en ce qui concerne les bibliothèques partagées, en mettant l'accent sur le linker et le loader.

## Slide [1, 2, 3] - Introduction au Linker

### Slide 1 

**Étape 1: Compilation**
- Source -> Fichiers objet
- Fichiers objets = conteneurs pour sections (code et données)

**Étape 2: Édition de liens**
- Rassemble les sections identiquement nommées de différents fichiers objets
- Les concatène en une grande section dans le binaire final
- Réordonne les sections pour les rendre adjacentes sur le disque si elles nécessitent les mêmes autorisations à l'exécution (formant des segments)

**Étape 3: Chargement**
- Au démarrage du programme, le loader mappe les segments vers des emplacements mémoire séparés
- Ajuste les autorisations de page en conséquence

### Slide 2

**Sections de code :**
- Contiennent généralement de nombreux appels de fonctions
- Dans le cas le plus simple, les appels sont internes au binaire lui-même
- En général, le processus charge également des bibliothèques partagées en mémoire

**Utilisation des bibliothèques partagées :**
- Un exécutable peut appeler une fonction implémentée dans une bibliothèque partagée
- Cette bibliothèque partagée peut elle-même appeler une fonction dans une autre bibliothèque partagée
- Le rôle principal du loader et de l'éditeur de liens est de connecter correctement ces appels

**Terminologie :**
- Sur Linux, les bibliothèques partagées sont appelées "Shared Objects" (SO(s))
- Sur Windows, elles sont appelées "Dynamic Link Libraries" (DLL(s))
- L'auteur utilisera le terme général "bibliothèque partagée" pour les deux
- Il utilisera également le terme "binaire" pour désigner à la fois un exécutable et une bibliothèque partagée
- La plupart de ce qui est dit s'applique aux deux.

### Slide 3

#### Vue naïve du câblage des appels de fonctions

**Exemple de code :**
- Si votre code inclut une fonction "foo" qui appelle "bar"
- "bar" n'est pas explicitement écrit dans la section de code du binaire lui-même

**Encodage dans le binaire :**
- L'appel à "bar" est encodé par une instruction de "call" avec une adresse nulle (placeholder)
- Le binaire contient également une section générée par l'éditeur de liens appelée "relock" pour les réallocations
- Les réallocations sont de petites instructions pour câbler les appels de la forme "trouve la fonction bar et remplace le placeholder par l'adresse trouvée"

## Slide 4 - Import data section

"Imports" désigne les bibliothèques que l'exécutable annonce qu'il doit importer, c'est-à-dire celles dont il dépend.
La section "idata" contient ces informations sur les imports.

## Slide 5 - Structure principale de la section "idata"

**Tableau de répertoires :**
- À l'intérieur de la section "idata", la structure principale est appelée "tableau de répertoires".
- Ce tableau contient des "entrées de répertoire", chaque paire correspondant à une DLL.

**Contenu des entrées de répertoire :**
- Chaque entrée de répertoire contient le nom de la DLL et une référence à une "table de recherche d'image".
- La table de recherche d'image est essentiellement une liste de noms ou d'indices de symboles à importer à partir de cette DLL.

**Résolution des symboles :**
- Le loader utilise les données de la table de recherche d'importation pour résoudre les symboles.
- Il place ensuite leurs adresses dans la "table d'adresses d'importation".
- Il y a une seule table d'adresses d'importation, mais chaque entrée de répertoire contient un décalage vers celle-ci.

## Slide 6 - Import Address Table - Création et utilisation des DLL sous Windows

**Exportation de fonctions :**
- Lorsque vous créez une DLL sous Windows et que vous exportez une fonction, vous utilisez la directive `__declspec(dllexport)` pour marquer la fonction.
- Cela rend la fonction disponible pour les autres binaires qui lient votre DLL.

**Section des exports :**
- Lorsqu'une fonction est exportée, une entrée est créée dans la section E4 (exports) de la DLL.
- Cette entrée est un "stub d'exportation" qui saute vers l'implémentation réelle de la fonction.

**Création du binaire d'importation :**
- Lorsque vous créez une DLL sous Windows, un autre binaire est également généré, appelé "fichier d'importation statique".
- Ce fichier est statique, ce qui signifie qu'il est destiné à être fusionné dans l'exécutable final.

**Fusion dans l'exécutable :**
- Le "linker" fusionne le fichier d'importation statique dans l'exécutable final.
- Cela ajoute au binaire final un "stub d'importation" qui saute simplement vers le "stub d'exportation" dans la DLL.
- Il ajoute également une entrée de répertoire dans la section "idata" pour cette DLL.

**Chargement par le loader :**
- Au moment du chargement, le loader utilise l'entrée dans la section "idata" pour localiser la DLL et la lier à l'exécutable.
- Il trouve les adresses appropriées et les écrit dans la "table d'adresses d'importation".

**Résolution des appels :**
- Lorsqu'une fonction est appelée depuis l'exécutable, l'appel est résolu par le "linker" en sautant d'abord vers la table d'imports.
- De là, il saute vers la table d'exports de la DLL, puis vers l'implémentation réelle de la fonction.

## Slide 7 - Interface Sémantique sous Windows

**Principe général :**
- Sous Windows, chaque binaire expose une interface par bibliothèque.
- Un binaire Windows indique au monde extérieur qu'il a besoin de trouver une certaine bibliothèque et d'utiliser les symboles qui y sont définis.
- Chaque bibliothèque est identifiée et utilisée de manière distincte.

## Slide 8 - Sections d'importation sous Linux

**Présence de deux sections d'importation :**
- Contrairement à Windows, un binaire Linux contient deux sections d'importation distinctes.
- La première section, appelée "dynamic section" (.dynamic), est simplement une liste de noms de bibliothèques.
- La deuxième section, appelée "symbol table" (.dynsim), contient des symboles spécifiques à partir de ces bibliothèques.

**Description schématique :**
- Schématiquement, la première section répertorie les noms des bibliothèques dont le binaire a besoin.
- La seconde section répertorie les symboles spécifiques nécessaires à partir de ces bibliothèques.

## Slide 9 - Interface sémantique sous Linux

**Annonce des bibliothèques nécessaires :**
- Un binaire Linux annonce séparément au monde extérieur les bibliothèques dont il a besoin.
- Il spécifie également quels symboles il doit utiliser à partir de chacune de ces bibliothèques.
- Cette communication ne se limite pas aux exécutables, mais s'applique également aux bibliothèques partagées.

**Ordre de recherche des symboles :**
- Lors de la résolution des symboles pour une bibliothèque partagée, par défaut, le binaire du programme est d'abord recherché pour ces symboles.
- Cela se fait avant même de chercher dans les bibliothèques dépendantes, et même avant la bibliothèque elle-même.

**Contrôle du comportement :**
- Ce comportement de recherche peut être contrôlé à l'aide de certains commutateurs de compilation.

## Slide 10 - Conclusion sur les différences sémantiques

**Sous Windows :**
- Un binaire Windows indique explicitement les bibliothèques qu'il souhaite importer et les symboles spécifiques qu'il veut utiliser depuis chaque bibliothèque.

**Sous Linux :**
- Un binaire Linux annonce les bibliothèques dont il a besoin et indique quels symboles doivent être recherchés à l'intérieur de ces bibliothèques, sans spécifier leur emplacement exact.

**Conséquences :**
- Cette approche sous Linux ouvre la voie à des possibilités intéressantes, mais peut entraîner des conflits si deux bibliothèques partagées implémentent le même symbole.
- Cependant, cela permet également une certaine flexibilité, permettant par exemple de substituer une fonction tierce par une implémentation personnalisée.


## Slide 11 - Introduction au "Position Independent Code"

**Formes de base des appels :**
- La forme la plus simple d'un appel est un appel à une adresse fixe, ce qui n'est pas "position independent".
- Une autre forme est un appel avec un décalage relatif par rapport au compteur de programme actuel, ce qui est "position independent".

**Utilisation de la "Global Offset Table" :**
- La forme principale de "position independent code" consiste à faire un appel à un décalage fixe dans la "Global Offset Table" (GOT).
- La GOT est une section générée au chargement par le loader et contient une liste d'adresses de fonctions résolues.
- L'appel est fait à l'adresse stockée dans cet emplacement de la GOT, assurant que l'appel est "position independent".

**Compréhension du positionnement :**
- Il est important de noter que si le binaire dans son ensemble n'est pas "position independent", la section de code l'est.
- Le "position independent code" était un objectif majeur dans la conception du système Linux, en particulier pour les bibliothèques partagées.
- Cela permet de partager les pages de code entre les processus qui utilisent les mêmes bibliothèques, économisant ainsi de la mémoire.

**Appels internes au binaire :**
- Ce flux d'appel s'applique également aux appels internes au binaire, pas seulement aux fonctions importées ou exportées.

## Slide 12 - Discussion sur le code position indépendant sous Linux

- Tous les codes des bibliothèques partagées sous Linux sont "position independent".
- Pour compiler du code "position independent", il faut spécifier un commutateur.
- Si un fichier objet n'a pas été compilé avec le commutateur adéquat, une erreur de liaison apparaît, demandant de recompiler avec l'option "-fpic".
- Il existe deux commutateurs similaires pour le positionnement indépendant (-fpic et -fPIC), mais ils diffèrent légèrement sur certaines architectures.
- Les exécutables ELF étaient historiquement chargés à l'adresse pour laquelle ils ont été construits, mais l'introduction de l'ASLR a permis de les charger à n'importe quelle adresse.

## Slide 13 - Temps de résolution

- Dans les exécutables Linux, la résolution des symboles est vérifiée au moment de l'édition des liens.
- Un exécutable Linux échouera à lier si certains symboles spécifiés ne sont pas trouvés dans les bibliothèques spécifiées.
- Cependant, pour les bibliothèques dynamiques, Linux peut les lier sans rechercher ces symboles manquants.
- Le commutateur qui contrôle ce comportement est "--no-allow-shlib-undefined".
- Il existe d'autres commutateurs plus délicats pour intervenir dans ce comportement, comme "--no-allow-sleep-undefined", mais leur fonctionnement n'est pas détaillé ici.

## Slide 14 - Résumé intermédiaire sur Linux

- Les dépendances en termes de symboles et de bibliothèques sont listées séparément sous Linux.
- Par défaut, tous les appels sont indirects et passent par la "Global Offset Table" (GOT).
- Par défaut, la résolution des symboles dans les bibliothèques est effectuée au chargement, et l'exécutable est recherché en premier.
- Ces faits ont des implications importantes pour les développeurs, dont certaines seront abordées dans la suite de la présentation.


## Slide 15 - Comment former un singleton à l'échelle du processus ?

- *Un singleton est une classe ou une structure de données qui ne peut être instanciée qu'une seule et unique fois.*<br><br><br>


- Sous Linux, pour former un singleton à l'échelle du processus, il suffit de placer l'implémentation dans l'exécutable lui-même.
- Partout où le singleton est utilisé, son implémentation est récupérée à partir de l'exécutable, où la définition est recherchée en premier.
- Sous Windows, la tâche est plus complexe : pour un singleton à l'échelle du processus, il faut planifier à l'avance et lier tous les binaires à une seule DLL implémentant le singleton.

## Slide 16 - Peut-on avoir des dépendances circulaires entre bibliothèques ?

- *Une dépendance circulaire entre bibliothèques signifie que la bibliothèque A dépend de la bibliothèque B, et que la bibliothèque B dépend de la bibliothèque A (ou tout autre cycle circulaire)*.<br><br><br>


- Sous Linux, il est possible d'avoir des dépendances circulaires entre bibliothèques sans besoin d'actions spéciales.
- Par défaut, les bibliothèques partagées sont construites avec une résolution des symboles retardée, ce qui permet à toutes les implémentations dans toutes les bibliothèques d'être disponibles pour le loader.
- En revanche, sous Windows, les dépendances circulaires entre bibliothèques ne sont pas autorisées par conception.

**Avis de l'auteur :**
Le présentateur exprime une préférence pour l'approche de Windows, estimant qu'elle encourage une meilleure organisation des bibliothèques partagées. Cette opinion est partagée par d'autres experts, comme Rui Ueyama, l'auteur de "LLD and MOLD linkers", qui suggère que l'utilisation des dépendances circulaires sous Linux est une mauvaise pratique.

## Slide 17 - Remplacement d'un symbole de bibliothèque partagée depuis un exécutable

- Sous Windows, il n'est pas possible de remplacer un symbole de bibliothèque partagée depuis un exécutable.
  - Les dépendances sont strictement définies et il n'y a pas de marge de manœuvre pour choisir quel symbole prendre dans quelle bibliothèque.

- Sous Linux, il est possible de remplacer un symbole de bibliothèque partagée depuis un exécutable.
  - Cela peut être réalisé en déclarant le symbole dans l'exécutable, ce qui interposera les autres implémentations de ce symbole, sauf si des commutateurs de compilation exotiques sont utilisés.

- Un exemple classique d'interposition de symboles est l'implémentation personnalisée des fonctions de gestion de la mémoire (comme new et delete) dans un programme C++.

- Sur Windows, les définitions de ces fonctions dans le programme ne peuvent pas être utilisées à la place des versions par défaut fournies par l'implémentation, ce qui n'est pas conforme à la norme C++.

## Slide 18 - Lazy Binding sur Windows et Linux

- Sous Windows, le lazy binding est appelé "delayed loading", et consiste à reporter la résolution des adresses de fonction jusqu'à leur première utilisation.
  - Cela permet d'éviter la résolution de milliers de fonctions inutilisées dans une grande bibliothèque, ce qui peut substantiellement réduire le temps de chargement.
  - Par défaut, le lazy binding n'est pas activé sur Windows, mais peut être ajouté avec un commutateur du linker pour spécifier une DLL à charger de manière différée.

- Sous Linux, le lazy binding est la norme et est activé par défaut.
  - Cela signifie que la résolution des adresses de fonction est reportée jusqu'à ce qu'elles soient effectivement appelées.
  - L'intervention dans ce comportement peut être réalisée avec la variable d'environnement LD_BIND_NOW, qui modifie le comportement à l'échelle du système.

## Slide 19 - Table de Liaison de Procédures (PLT)

- La PLT est une section exécutable générée par le loader sous Linux, qui joue un rôle crucial dans la résolution des appels de fonctions.
- Initialement, un appel est fait à un décalage fixe dans la PLT, où la première instruction est un saut vers un emplacement fixe dans la GOT.
- La GOT contient initialement l'adresse d'un lieu qui effectue la résolution réelle de la fonction.
- Le loader, toujours chargé dans le processus, effectue cette résolution lors du premier appel à la fonction et met à jour l'adresse dans la GOT avec la vraie adresse de la fonction.
- À partir du deuxième appel, l'appel est fait directement à l'implémentation réelle de la fonction, en passant par la PLT et la GOT.

## Slide 20 - Comparaison de pointeurs de fonction (Linux)

>***C++ standard :*** *Si deux pointeurs de fonction sont nuls ou pointent vers la même fonction, ils sont considérés comme égaux.*

- Sous Linux, la comparaison de pointeurs de fonction est possible grâce à une section importante appelée ".symtab" (table des symboles).
- Cependant, les appels réels sont faits à des adresses dans des PLT différentes pour les exécutables et les bibliothèques partagées.
- Pour permettre la comparaison de pointeurs de fonction, le loader remplit la table des symboles avec l'adresse de la PLT pour chaque fonction lors du chargement.
- Cela permet de comparer les pointeurs de fonction, même s'ils pointent vers des adresses différentes dans la PLT.

## Slide 21 - Comparaison de pointeurs de fonction (Windows)

- Sous Windows, avec les DLL chargées de manière différée, l'adresse d'une fonction avant son appel diffère de son adresse après son appel.
- Cela signifie que deux adresses de fonction pointant vers la même fonction peuvent être considérées comme différentes avant et après l'appel.
- Par conséquent, sous Windows, un pointeur de fonction n'est pas égal à lui-même dans le même binaire, et encore moins à un pointeur de fonction dans des binaires différents.
- En présence de chargements différés, Windows ne respecte pas la norme de comparaison de pointeurs de fonction telle que définie par le standard C++.

## Slide 22 - Visibilité des symboles (Windows)

- Sous Windows, la visibilité des symboles est contrôlée par différentes directives de compilation.
- On peut utiliser "__declspec(export)" pour marquer les symboles à exporter, ce qui les ajoute à la section "idata" de l'exécutable.
- On peut également utiliser "__declspec(import)" pour marquer les symboles à importer, mais cela ne les ajoute pas à la section "idata".
- Par défaut, un symbole n'est ni exporté ni importé, et est considéré comme interne au binaire.

## Slide 23 - Visibilité des symboles (Linux)

- Sous Linux, il n'y a pas de directives analogues a import et export.
- La distinction se fait entre les symboles publics et cachés.
- Les symboles publics passent par le PLT et la GOT, et sont visibles par d'autres bibliothèques et exécutables.
- Par défaut, la visibilité des symboles est publique, ce qui signifie qu'ils sont à la fois importés et exportés.
- On peut modifier la visibilité avec des directives comme "fvisibility=hidden" pour les rendre cachés.
- Cette fonctionnalité peut améliorer considérablement les temps de liaison et de chargement des bibliothèques partagées et est fortement recommandée.
- Les symboles cachés ont été ajoutés tardivement à la conception d'ELF et leur par le manuel de GCC (GNU Compiler Collection, le compilateur C++ standard sous Linux).

## Slide 24 - Loader

- Le loader est une entité qui peut avoir plusieurs noms comme "Dynamic Linker", "Interpreter", "Image Loader"...
- Sous Linux, il s'agit d'un fichier appelé "ld-linux-x86-64.so.2".
- Sur Windows, le chargeur est une composante logique qui regroupe toutes les API dans ntdll commençant par "ldr".
- Son rôle principal est de charger les bibliothèques partagées dans l'espace d'adressage utilisateur et de remplacer certaines entrées dans les tables de ce dernier.

## Slide 25 - C++ and shared libs

Dans cette derniére diapositive, Ofek Shilon explique le manque de consideration du standart C++ pour les bibliothèques partagées, et explique que cela rend le standart peu pratique dans la réalité. Il indique que, selon lui, il faudrait supprimer les clauses liées aux bibliothèques partagées du standard pour éviter des déclarations inapplicables et propose également de revoir la définition des notions de programme et d'implémentation.

## Conclusion

Durant la conclusion de sa conference, Ofek Shilon indique que plusieurs sujets, tels que la "relocation", ou le "versioning" ont été passé sous silence. Il remercie également Michael Kerrisk, qui a commenté la premiere ébauche de sa présentation. Il finit sa conférance en répondant aux questions des personnes présentes, mais cette partie n'est pas partagé dans la rediffusion.